<a href="https://colab.research.google.com/github/yorique/dashboard-bacias-ufra/blob/main/contrato001.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')
print("Drive acessível:", Path('/content/drive/MyDrive').is_dir())

Mounted at /content/drive
Drive acessível: True


In [5]:
from google.auth import default
from google.auth.transport.requests import AuthorizedSession

artifact_ids = {
    "codigo_bloco10_v2": "1VdWc3flW7KcPpt79hQjwiIJZMUQIzUt1",
    "codigo_bloco22_v2": "1Pt-LtiLa_RtvIRha6jZ93VTvAJJzobdg",
    "relatorio_bloco10": "1uW42Lsl-LeL5UWVMPY4Oj7Yj7dvWy5Gw",
    "relatorio_bloco22": "1oblWLSlv-AnsfcNV75KfjxDSWyAI5kP9",
    "configuracao_bloco10": "1XgiJ7stTQ9m8ZmYlaaYyu1JhaO7veNr6",
    "configuracao_bloco22": "1BRX3HBzIw79ETuAGDrdbJEg9QKhW9NaD",
}
fields = "id,name,mimeType,createdTime,modifiedTime,size,md5Checksum,version,description,parents,trashed"
creds, _ = default(scopes=["https://www.googleapis.com/auth/drive.readonly"])
session = AuthorizedSession(creds)
metadata = {}
for label, file_id in artifact_ids.items():
    response = session.get(
        f"https://www.googleapis.com/drive/v3/files/{file_id}",
        params={"fields": fields},
        timeout=30,
    )
    if response.ok:
        metadata[label] = response.json()
    else:
        metadata[label] = {"http_status": response.status_code, "error": response.text[:300]}
for label, record in metadata.items():
    print(label, record)

codigo_bloco10_v2 {'id': '1VdWc3flW7KcPpt79hQjwiIJZMUQIzUt1', 'name': 'Bloco_10_V2_CHIRPS_Downscale.txt', 'mimeType': 'text/plain', 'trashed': False, 'parents': ['1ynPR4xMd_yVwVfLDLSK4PaSjI0AJb4e3'], 'version': '6', 'createdTime': '2026-11-03T16:48:03.311Z', 'modifiedTime': '2026-11-03T16:49:41.190Z', 'md5Checksum': '70812aec1902376a771bed54abf6b05a', 'size': '120254'}
codigo_bloco22_v2 {'id': '1Pt-LtiLa_RtvIRha6jZ93VTvAJJzobdg', 'name': 'Bloco_22_V2_Interpretacao_Envelopes_Extrapolacao_Multibacias.txt', 'mimeType': 'text/plain', 'trashed': False, 'parents': ['1ynPR4xMd_yVwVfLDLSK4PaSjI0AJb4e3'], 'version': '6', 'createdTime': '2026-11-03T16:48:10.003Z', 'modifiedTime': '2026-11-03T16:49:42.884Z', 'md5Checksum': '35bade3f06556f635c35509fc912185a', 'size': '83987'}
relatorio_bloco10 {'id': '1uW42Lsl-LeL5UWVMPY4Oj7Yj7dvWy5Gw', 'name': 'Relatorio_Bloco10_V2_SHAP_Permutacao_Caete.txt', 'mimeType': 'text/plain', 'trashed': False, 'parents': ['1luRdau8GfF95X3Rl6fEQvwCPZ5Z2zqki'], 'version': 

In [4]:
from google.colab import auth

auth.authenticate_user()
print("Autenticação do Colab concluída.")

Autenticação do Colab concluída.


In [6]:
import hashlib

artifacts = {}
for label, file_id in artifact_ids.items():
    response = session.get(
        f"https://www.googleapis.com/drive/v3/files/{file_id}",
        params={"alt": "media"},
        timeout=60,
    )
    response.raise_for_status()
    payload = response.content
    artifacts[label] = payload
    print(label, "bytes=", len(payload), "sha256=", hashlib.sha256(payload).hexdigest())

codigo_bloco10_v2 bytes= 120254 sha256= 2f72a368f55a9b604e1e75a4e058671ba4d13e299340e25536942dba77fdd5fd
codigo_bloco22_v2 bytes= 83987 sha256= 176235c0e33d4600f6cad79aea87353c10b9f72d68506fd6810a29c197da5225
relatorio_bloco10 bytes= 3842 sha256= d042772467a20ec0cfe4340f2aacd20ff55aab267c3e22a95707d5113a640643
relatorio_bloco22 bytes= 5735 sha256= a57146f7d9b3d88e31a328b204f4d83403f66ec0f37d2d96ba658c61ef37fac5
configuracao_bloco10 bytes= 23108 sha256= ec356d79bada6cbc1bf3d500bbce125e91414270923b7e75ff654cad69a127c0
configuracao_bloco22 bytes= 3151 sha256= 387ed63ce03cc02f4f39ba9fe63c5c0df47cf3b0ffc4c4c3173403b6a32174c7


In [7]:
folder_ids = sorted({record["parents"][0] for record in metadata.values() if "parents" in record})
folder_inventory = {}
for folder_id in folder_ids:
    response = session.get(
        "https://www.googleapis.com/drive/v3/files",
        params={
            "q": f"'{folder_id}' in parents and trashed = false",
            "fields": "files(id,name,mimeType,createdTime,modifiedTime,size,md5Checksum,version,parents)",
            "pageSize": 1000,
        },
        timeout=30,
    )
    response.raise_for_status()
    files = response.json().get("files", [])
    folder_inventory[folder_id] = files
    print(f"PASTA {folder_id} | {len(files)} item(ns)")
    for item in sorted(files, key=lambda x: x["name"].lower()):
        print(item)

PASTA 1CHgAwU62Qdu23SA8-iPkhOTQhxBnyhka | 28 item(ns)
{'md5Checksum': 'd32b0783c678fcffb70d958760371ee2', 'mimeType': 'application/json', 'parents': ['1CHgAwU62Qdu23SA8-iPkhOTQhxBnyhka'], 'size': '23108', 'id': '1XgiJ7stTQ9m8ZmYlaaYyu1JhaO7veNr6', 'name': 'Configuracao_Bloco10_V2_SHAP_Permutacao_Caete.json', 'createdTime': '2026-08-16T13:21:26.956Z', 'modifiedTime': '2026-08-16T13:21:26.964Z', 'version': '2'}
{'md5Checksum': '554aae6d76c430b7f7db7ca7fefb3dbf', 'mimeType': 'application/json', 'parents': ['1CHgAwU62Qdu23SA8-iPkhOTQhxBnyhka'], 'size': '5222', 'id': '12ZhSqRMnykpdqv42Lj1HLQvoGZD4vqGn', 'name': 'Configuracao_Bloco11_V2_Predicao_Bruta_Caete.json', 'createdTime': '2026-08-16T15:34:57.900Z', 'modifiedTime': '2026-08-16T15:34:57.900Z', 'version': '3'}
{'md5Checksum': 'e7fd2e8f44e9c16c9375b89ba990b731', 'mimeType': 'application/json', 'parents': ['1CHgAwU62Qdu23SA8-iPkhOTQhxBnyhka'], 'size': '3963', 'id': '1Ivf334iGkVj9lIhjvyjL3gg8hhjJFtop', 'name': 'Configuracao_Bloco12_V2_Corr

In [8]:
import csv, io, json

provenance_ids = {
    "manifesto_bloco10": "17tLSzQjk-NkNgvNtbdCVSGo2gFsjEu_P",
    "manifesto_bloco22": "1ltb1quO324t0f4sep9Uh9jsg70Mk2i1i",
    "inventario_bloco22": "1psvMKls1FKyQLYZRTG4gT3oPuFqwZF6j",
    "auditoria_b21_b22": "1PE38qduYSVYMehMEDpckpcfW4CNOteSr",
}
provenance_bytes = {}
for label, file_id in provenance_ids.items():
    response = session.get(
        f"https://www.googleapis.com/drive/v3/files/{file_id}",
        params={"alt": "media"}, timeout=60,
    )
    response.raise_for_status()
    provenance_bytes[label] = response.content
    print(label, "sha256=", hashlib.sha256(response.content).hexdigest())

for label, file_id in {k: artifact_ids[k] for k in ("codigo_bloco10_v2", "codigo_bloco22_v2")}.items():
    response = session.get(
        f"https://www.googleapis.com/drive/v3/files/{file_id}/revisions",
        params={"fields": "revisions(id,modifiedTime,md5Checksum,size,originalFilename,keepForever,published)"},
        timeout=30,
    )
    print("REVISOES", label, response.status_code, response.json() if response.ok else response.text[:300])

pipeline_id = "1bPAVzHo5Ax0HCYVd2O_8eoTrqiYtoPCm"
response = session.get(
    f"https://www.googleapis.com/drive/v3/files/{pipeline_id}",
    params={"alt": "media"}, headers={"Range": "bytes=0-4095"}, timeout=30,
)
print("PIPELINE_PREFIX", response.status_code, response.headers.get("Content-Range"), response.content[:200])

manifesto_bloco10 sha256= d0350dcb80a7f779a7c8c3ae11aa074c82ed4638e3801871e07e53ab6abe303a
manifesto_bloco22 sha256= 0a71be1091d6d885960cab874a3d5205e5c49a340f84b268f226b6a8a7a09c6f
inventario_bloco22 sha256= d0f2a847f837ae1da674aef30a8d29e8817db52fd16f769ca1985dc1a3b44c96
auditoria_b21_b22 sha256= bd7f8b01ff3007f33f5b4a13d70f4000a1bfb37d18335f084ad3974b124c5876
REVISOES codigo_bloco10_v2 200 {'revisions': [{'id': '0B-HoCUesTpISSEVYQ2NMTnNsaGpvYlk5UzZKZWZyazN0MCtFPQ', 'modifiedTime': '2026-08-16T13:07:31.118Z', 'keepForever': True, 'published': False, 'originalFilename': 'Bloco_10_V2_CHIRPS_Downscale.txt', 'md5Checksum': '70812aec1902376a771bed54abf6b05a', 'size': '120254'}, {'id': '0B-HoCUesTpISWlVXa1g1OGN5azZLOERsSmUxQmFOUXhKdSswPQ', 'modifiedTime': '2026-11-03T16:49:41.190Z', 'keepForever': True, 'published': False, 'originalFilename': 'Bloco_10_V2_CHIRPS_Downscale.txt', 'md5Checksum': '70812aec1902376a771bed54abf6b05a', 'size': '120254'}]}
REVISOES codigo_bloco22_v2 200 {'revisions

In [9]:
def get_media(file_id, revision_id=None):
    url = f"https://www.googleapis.com/drive/v3/files/{file_id}"
    if revision_id:
        url += f"/revisions/{revision_id}"
    response = session.get(url, params={"alt": "media"}, timeout=120)
    response.raise_for_status()
    return response.content

historical_revisions = {
    "bloco10_agosto": (artifact_ids["codigo_bloco10_v2"], "0B-HoCUesTpISSEVYQ2NMTnNsaGpvYlk5UzZKZWZyazN0MCtFPQ"),
    "bloco22_agosto": (artifact_ids["codigo_bloco22_v2"], "0B-HoCUesTpISaDBCY0V6emdqZElmc2p1bGdrWEJFa0U5bjNrPQ"),
}
for label, (file_id, revision_id) in historical_revisions.items():
    payload = get_media(file_id, revision_id)
    print("REVISAO_HISTORICA", label, "bytes=", len(payload), "sha256=", hashlib.sha256(payload).hexdigest())

for label in ("manifesto_bloco10", "manifesto_bloco22", "inventario_bloco22", "auditoria_b21_b22"):
    text = provenance_bytes[label].decode("utf-8-sig", errors="replace")
    reader = csv.DictReader(io.StringIO(text))
    rows = list(reader)
    fields = reader.fieldnames or []
    selected = [f for f in fields if any(token in f.lower() for token in ("hash", "sha", "codigo", "code", "arquivo", "file", "vers", "data", "date", "notebook", "script", "exec", "modelo", "model", "origem", "source"))]
    print("CSV", label, "linhas=", len(rows), "campos_relevantes=", selected)
    for row in rows:
        print({f: row.get(f) for f in selected})

for label in ("relatorio_bloco10", "relatorio_bloco22"):
    print("RELATORIO", label)
    print(artifacts[label].decode("utf-8", errors="replace"))

for label in ("configuracao_bloco10", "configuracao_bloco22"):
    config = json.loads(artifacts[label].decode("utf-8"))
    print("CONFIG", label, json.dumps(config, ensure_ascii=False, indent=2))

pipeline_bytes = get_media(pipeline_id)
pipeline = json.loads(pipeline_bytes.decode("utf-8"))
print("NOTEBOOK", {"id": pipeline_id, "sha256": hashlib.sha256(pipeline_bytes).hexdigest(), "nbformat": pipeline.get("nbformat"), "metadata": pipeline.get("metadata", {}), "cells": len(pipeline.get("cells", []))})
current_sources = {
    "bloco10": artifacts["codigo_bloco10_v2"].decode("utf-8", errors="replace"),
    "bloco22": artifacts["codigo_bloco22_v2"].decode("utf-8", errors="replace"),
}
for target, source_text in current_sources.items():
    normalized_target = "\n".join(line.rstrip() for line in source_text.splitlines()).strip()
    exact = []
    normalized = []
    markers = []
    for index, cell in enumerate(pipeline.get("cells", [])):
        cell_source = "".join(cell.get("source", []))
        if cell_source == source_text:
            exact.append(index)
        if "\n".join(line.rstrip() for line in cell_source.splitlines()).strip() == normalized_target:
            normalized.append(index)
        if target.replace("bloco", "bloco ") in cell_source.lower() or ("shap" in cell_source.lower() and ("10" in cell_source or "22" in cell_source)):
            markers.append({"index": index, "type": cell.get("cell_type"), "execution_count": cell.get("execution_count"), "executionInfo": cell.get("metadata", {}).get("executionInfo"), "source_sha256": hashlib.sha256(cell_source.encode()).hexdigest(), "chars": len(cell_source)})
    print("COMPARACAO_NOTEBOOK", target, {"celdas_exatas": exact, "celdas_normalizadas": normalized, "celdas_marcadas": markers[:30]})

REVISAO_HISTORICA bloco10_agosto bytes= 120254 sha256= 2f72a368f55a9b604e1e75a4e058671ba4d13e299340e25536942dba77fdd5fd
REVISAO_HISTORICA bloco22_agosto bytes= 83987 sha256= 176235c0e33d4600f6cad79aea87353c10b9f72d68506fd6810a29c197da5225
CSV manifesto_bloco10 linhas= 25 campos_relevantes= ['Arquivo;Caminho;Existe;Tamanho_bytes;SHA256']
{'Arquivo;Caminho;Existe;Tamanho_bytes;SHA256': 'Auditoria_Retomada_Drive_Bloco10_V2.csv;/content/drive/MyDrive/Meus arquivos/UFRA/GEE/TCC - CHIRPS - V2/Auditorias/Bloco 10 V2 - SHAP Permutacao/Tabelas/Auditoria_Retomada_Drive_Bloco10_V2.csv;True;1376;a956fef425b703de6781bb48c518948cb134a7c7d22a5574aa125069a755c8ec'}
{'Arquivo;Caminho;Existe;Tamanho_bytes;SHA256': 'Auditoria_Manifesto_Bloco9_no_Bloco10_V2.csv;/content/drive/MyDrive/Meus arquivos/UFRA/GEE/TCC - CHIRPS - V2/Auditorias/Bloco 10 V2 - SHAP Permutacao/Tabelas/Auditoria_Manifesto_Bloco9_no_Bloco10_V2.csv;True;9258;c0cbe5e41cd61ebfe4c8188324fcf1ef55fe43f6bc0237e0d178e07a774a5873'}
{'Arquivo;Cam

In [10]:
from datetime import datetime, timezone
from zoneinfo import ZoneInfo

pipeline_meta_response = session.get(
    f"https://www.googleapis.com/drive/v3/files/{pipeline_id}",
    params={"fields": "id,name,mimeType,createdTime,modifiedTime,size,md5Checksum,version"}, timeout=30,
)
print("METADADOS_NOTEBOOK", pipeline_meta_response.json())

for index, label in ((9, "BLOCO10"), (22, "BLOCO22")):
    cell = pipeline["cells"][index]
    info = cell.get("metadata", {}).get("executionInfo", {})
    timestamp = info.get("timestamp")
    when = None if timestamp is None else datetime.fromtimestamp(timestamp / 1000, tz=timezone.utc).astimezone(ZoneInfo("America/Sao_Paulo")).isoformat()
    source = "".join(cell.get("source", []))
    output_text = []
    for output in cell.get("outputs", []):
        if "text" in output:
            text = output["text"]
            output_text.append("".join(text) if isinstance(text, list) else str(text))
        if "data" in output and "text/plain" in output["data"]:
            value = output["data"]["text/plain"]
            output_text.append("".join(value) if isinstance(value, list) else str(value))
    joined_output = "\n".join(output_text)
    relevant_output_lines = [line for line in joined_output.splitlines() if any(token in line.lower() for token in ("salv", "relat", "config", "manifest", "shap", ".csv", ".json", ".txt", "teste"))]
    relevant_source_lines = [line.strip() for line in source.splitlines() if any(token in line.lower() for token in ("versao", "versão", "config", "relatorio", "manifest", "shap", "bloco_10", "bloco_22"))]
    print("CELULA", label, {"indice": index, "status": info.get("status"), "horario_brt": when, "elapsed_ms": info.get("elapsed"), "source_sha256": hashlib.sha256(source.encode()).hexdigest(), "outputs": len(cell.get("outputs", [])), "output_chars": len(joined_output)})
    print("SAIDA_RELEVANTE", relevant_output_lines[:40])
    print("FONTE_RELEVANTE", relevant_source_lines[:40])

for label in ("manifesto_bloco10", "manifesto_bloco22"):
    text = provenance_bytes[label].decode("utf-8-sig", errors="replace")
    reader = csv.DictReader(io.StringIO(text), delimiter=";")
    rows = list(reader)
    print("MANIFESTO", label, {"campos": reader.fieldnames, "linhas": len(rows)})
    for row in rows:
        filename = (row.get("Arquivo") or "").lower()
        if any(token in filename for token in ("configuracao", "relatorio", "manifesto", "shap", "teste")):
            print(row)

METADADOS_NOTEBOOK {'id': '1bPAVzHo5Ax0HCYVd2O_8eoTrqiYtoPCm', 'name': 'Pipeline CHIRPS V2', 'mimeType': 'application/octet-stream', 'version': '2', 'createdTime': '2026-11-03T16:49:29.557Z', 'modifiedTime': '2026-08-21T16:14:01.427Z', 'md5Checksum': 'e0dbde955734e49debe7dd87e85269a9', 'size': '4552640'}
CELULA BLOCO10 {'indice': 9, 'status': 'ok', 'horario_brt': '2026-08-16T10:21:26.454000-03:00', 'elapsed_ms': 810754, 'source_sha256': '2f72a368f55a9b604e1e75a4e058671ba4d13e299340e25536942dba77fdd5fd', 'outputs': 1, 'output_chars': 4177}
SAIDA_RELEVANTE ['BLOCO 10 V2 - SHAP, PERMUTACAO E CORRELACAO', '  Manifesto Bloco 9 no Drive: 27/27 produtos integros', '  RMSE teste final: 55.696008', '  R2 teste final: 0.879569', '  PBIAS teste final: 13.819571%', 'SHAP NO TESTE FINAL 2020-2024', '  Calibracao SHAP: Spearman=0.9886; top10=100.00%; tempo=592.0s', '  Tempo SHAP principal: 1.1s', 'IMPORTANCIA POR PERMUTACAO - TESTE FINAL 2020-2024', 'Manifesto Bloco 9 auditado no Drive: 27/27', 'SHA